# Nettoyage de la Data
## Infos : 
__Fichiers nécessaires :__
- lap_times.csv
- races.csv (on trie les GP, on garde année > 1997)
- circuits.csv (tri par circuit, on garde que les 5 qu'on étudie)
- f1db-races.csv (longueur des circuits -> calculer avg_speed sur un tour)
- results.csv (récupérer le statut de la course pour chaque pilote, on garde que les lignes )

__Fichiers sortants :__
- resultats_laps.csv
- grand_prix.csv

## Initialistaion

In [ ]:
import pandas as pd

# On garde seulement les colonnes qui nous sont utiles pour alléger et optimiser
colonnes_voulues = ["raceId","driverId","lap","time","milliseconds"]
df_times = pd.read_csv("../data/02_silver/lap_times.csv", usecols=colonnes_voulues)

df_times.shape #test

## Renommage des colonnes
On renomme les colonnes pour qu'on se conforme au dictionnaire.

In [ ]:
df_times = df_times.rename(columns={
    "raceId": "id_grand_prix",
    "driverId": "id_pilote",
    "lap": "id_lap",
    "time": "lap_time"
})

df_times.head()

## Fusion de cette table avec races.csv
Pour pouvoir filtrer par période (on étudie seulement la période 1997-2026), on va avoir besoin de la table races.csv qui répertories pour chaque GP, leur nom (ca sera plus facile pour les identifier) mais surtout leur année.

In [ ]:
import matplotlib.pyplot as plt

# Import de la 2e table
colonnes_voulues = ["raceId","year","circuitId","date"]
df_races = pd.read_csv("../data/02_silver/races.csv", usecols=colonnes_voulues)

#Pareil que pour la 1e, on renomme les colonnes
df_races = df_races.rename(columns={
    "raceId": "id_grand_prix",
    "circuitId": "id_circuit"
})

df_fusion = pd.merge(
    df_times,
    df_races,
    left_on="id_grand_prix",
    right_on="id_grand_prix",
    how="left"
)

display(df_fusion.sort_values(["year"]).head())

# Vérification de la distribution des années sur chaque ligne de lap_times.csv
distrib_annees = df_fusion["year"].value_counts().sort_index()

plt.figure(figsize=(12, 5))
plt.bar(distrib_annees.index, distrib_annees.values, color="#E10600", width=0.8)

plt.title("Distribution des données par année")
plt.xlabel("Année")
plt.ylabel("Nombre de lignes")
plt.grid(axis="y", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

# On peut voir qu'on a qu'une seule année à retirer pour que ce fichier soit conforme à notre scope défini

## Filtrage par année
On peut donc maintenant filtrer les lignes de lap_times par année

In [ ]:
df_fusion = df_fusion[df_fusion["year"] >= 1997]

display(df_fusion.sort_values(["year"]).head())

df_fusion.shape

## Fusion avec circuits.csv
On veut seulement les lap_times des circuits concernés, c'est à dire ceux présents dans le fichier gold circuits.csv, on le fusionne donc avec les lap_times.

In [ ]:
df_circuits = pd.read_csv("../data/03_gold/circuits.csv")

df_fusion = pd.merge(
    df_fusion,
    df_circuits,
    left_on="id_circuit",
    right_on="id_circuit",
    how="right"
)

display(df_fusion.shape)
display(df_fusion.head())

# Suppression des colonnes non-pertinentes
Dans notre future table `resultats_laps`, on a seulement besoin de `id_grand_prix`, `id_pilote`, `id_lap`, `lap_time`

In [ ]:
df_fusion = df_fusion[["id_grand_prix", "id_pilote", "id_lap", "lap_time","milliseconds","date"]]

display(df_fusion.shape)
display(df_fusion[~df_fusion.duplicated(subset=["id_grand_prix"])].sort_values("id_grand_prix").head()) # Apercu de la table avec plusieurs GP

## Fusion avec f1db-races
Pour calculer la avg_speed, on va diviser la longueur d'un tour par le chrono que le pilote a fait pendant un tour. Il nous manque la longueur, on va donc la chercher dans ce .csv. Avant de fusionner, on nettoie vite fait le fichier.

### Nettoyage

In [ ]:
import pandas as pd

df_f1db_races = pd.read_csv("../data/02_silver/f1db-races.csv")

# Tri des colonnes inutiles
df_f1db_races = df_f1db_races[["id","year","round","circuitId","courseLength","date"]]

display(df_f1db_races.shape)

# Tri par circuit
df_f1db_races = df_f1db_races[df_f1db_races["circuitId"].isin(["suzuka","monza","monaco","spa-francorchamps","silverstone"])]

display(df_f1db_races.shape)

# Tri par année
df_f1db_races = df_f1db_races[df_f1db_races["year"] >= 1997]

# Vérif nombre gp
print(f"Nombre de GPs réel (après tri) : {df_f1db_races.shape[0]} \nNombre de GPs théorique (5 tous les ans) : {(2026 - 1997 + 1) * 5} \nOn peut voir une différence de {(2026 - 1997 + 1) * 5 - df_f1db_races.shape[0]} GPs, qui est affichée plus en détails ci-dessous.")

df_incomplets = df_f1db_races.groupby("year").filter(lambda g: len(g) != 5)
display(df_incomplets.sort_values(["year", "round"]))

### Fusion
Les IDs des GP ne correspondent pas forcément (on travaille sur 2 csv provenant de datasets différents), on va avoir besoin de la colonne date (date à laquelle le GP s'est déroulé) pour faire le join entre ces deux csv.

In [ ]:
display(df_fusion.shape)

df_fusion = pd.merge(
    df_fusion,
    df_f1db_races[["id", "date", "year", "courseLength"]],
    left_on="date",
    right_on="date",
    how="left"
).drop(columns=["id_grand_prix"]
).rename(columns={
    "id": "id_grand_prix",
})

display(df_fusion.shape)
display(df_fusion.head())

## Fusion avec results pour récupérer le status
On va trier les lignes par le statut de la course du pilote. Il y a cependant un petit problème de grain, notre table a pour l'instant un grain de 1 ligne = un tour par pilote alors que le fichier qu'on va fusionner a plutôt un grain de 1 ligne = 1 course par pilote. On décide de virer tous les tours d'un pilote sur une course si celle-ci a un statut autre que 1 et de 11 à 19 (1 étant "Finished", course finie sans panne ni autres problèmes et de 11 à 19, correspondant à course terminée mais avec 1-9 tour(s) de retard)

In [ ]:
df_results = pd.read_csv("../data/02_silver/results.csv", usecols=["raceId", "driverId", "statusId"]) # On prend que le colonnes pertinentes

df_fusion = pd.merge(
    df_fusion,
    df_results,
    left_on=["id_grand_prix", "id_pilote"],
    right_on=["raceId", "driverId"],
    how="left"
).drop(columns=["raceId", "driverId"] # On retire les colonnes en doublon
).rename(columns={
    "date": "date_grand_prix",
    "courseLength": "lap_length",
    "year": "id_saison",
    "statusId": "id_status" # FLOAT AU LIEU DE INT
}) # On se conforme aux noms dans la BDD

df_fusion.head()

## Export modifications vers des fichiers .csv

### Table resultats_laps

In [ ]:
df_resultats_laps = df_fusion[["id_grand_prix", "id_pilote", "id_lap", "id_status", "lap_time", "lap_length"]]

#id ecurie a faire

df_resultats_laps.to_csv("../data/03_gold/resultats_laps.csv", index=False, encoding="utf-8")

### Table grand_prix

In [ ]:
df_grand_prix = (
    df_fusion[["id_grand_prix", "id_saison", "date_grand_prix"]]
    .drop_duplicates()
    .sort_values("date_grand_prix")
    .reset_index(drop=True)
)

#nom_grand_prix et date_grand_prix et id_circuit a faire

df_grand_prix.to_csv("../data/03_gold/grand_prix.csv", index=False, encoding="utf-8")